# Pathway analysis    ---- UNCHANGED FROM YO all

In [1]:
import scanpy as sc
import decoupler as dc

# Only needed for processing
import numpy as np
import pandas as pd
import pandas as pd
import yaml

In [2]:
dds_files_path = '../data/dds_files.yml'

In [3]:
with open(dds_files_path, 'r') as file:
    dds_dict = yaml.safe_load(file)

In [4]:
df_dds_dict = {}
df_sig_dict = {}
df_de_dict = {}
pvalue = 0.05
elbow = 2
for dds, file in dds_dict.items():
    df = pd.read_csv(file, index_col=0)
    df_dds_dict[dds] =  df.copy()
    df_sig_dict[dds] = df[df['padj']<pvalue]
    dds_de = df[(abs(df['log2FoldChange'])>elbow) & (df['padj']<pvalue)]
    df_de_dict[dds] = dds_de.copy()


In [5]:
'/home/amore/work/data/RNAseq_abundances_adjusted_combat_inmose_young.vs.old_DDS.csv'

'/home/amore/work/data/RNAseq_abundances_adjusted_combat_inmose_young.vs.old_DDS.csv'

In [6]:
results_df =dds

## Get msigdb

In [7]:
msigdb = dc.get_resource('MSigDB')
msigdb# Sort the results by the ranking metric (e.g., log2 fold change)


0.00B [00:00, ?B/s]

0.00B [00:00, ?B/s]

,genesymbol,collection,geneset
0,MAFF,chemical_and_genetic_perturbations,BOYAULT_LIVER_CANCER_SUBCLASS_G56_DN
1,MAFF,chemical_and_genetic_perturbations,ELVIDGE_HYPOXIA_UP
2,MAFF,chemical_and_genetic_perturbations,NUYTTEN_NIPP1_TARGETS_DN
3,MAFF,immunesigdb,GSE17721_POLYIC_VS_GARDIQUIMOD_4H_BMDC_DN
4,MAFF,chemical_and_genetic_perturbations,SCHAEFFER_PROSTATE_DEVELOPMENT_12HR_UP
...,...,...,...
3838543,PRAMEF22,go_biological_process,GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PR...
3838544,PRAMEF22,go_biological_process,GOBP_APOPTOTIC_PROCESS
3838545,PRAMEF22,go_biological_process,GOBP_REGULATION_OF_CELL_DEATH
3838546,PRAMEF22,go_biological_process,GOBP_NEGATIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS


In [8]:
# Filter by hallmark
#msigdb = msigdb[msigdb['collection']=='hallmark']

# Remove duplicated entries
msigdb = msigdb[~msigdb.duplicated(['geneset', 'genesymbol'])]

# Rename
#msigdb.loc[:, 'geneset'] = [name.split('HALLMARK_')[1] for name in msigdb['geneset']]

msigdb

,genesymbol,collection,geneset
0,MAFF,chemical_and_genetic_perturbations,BOYAULT_LIVER_CANCER_SUBCLASS_G56_DN
1,MAFF,chemical_and_genetic_perturbations,ELVIDGE_HYPOXIA_UP
2,MAFF,chemical_and_genetic_perturbations,NUYTTEN_NIPP1_TARGETS_DN
3,MAFF,immunesigdb,GSE17721_POLYIC_VS_GARDIQUIMOD_4H_BMDC_DN
4,MAFF,chemical_and_genetic_perturbations,SCHAEFFER_PROSTATE_DEVELOPMENT_12HR_UP
...,...,...,...
3838543,PRAMEF22,go_biological_process,GOBP_POSITIVE_REGULATION_OF_CELL_POPULATION_PR...
3838544,PRAMEF22,go_biological_process,GOBP_APOPTOTIC_PROCESS
3838545,PRAMEF22,go_biological_process,GOBP_REGULATION_OF_CELL_DEATH
3838546,PRAMEF22,go_biological_process,GOBP_NEGATIVE_REGULATION_OF_DEVELOPMENTAL_PROCESS


In [9]:
set(msigdb["collection"])

{'biocarta_pathways',
 'cancer_gene_neighborhoods',
 'cancer_modules',
 'cell_type_signatures',
 'chemical_and_genetic_perturbations',
 'go_biological_process',
 'go_cellular_component',
 'go_molecular_function',
 'hallmark',
 'human_phenotype_ontology',
 'immunesigdb',
 'kegg_pathways',
 'mirna_targets_legacy',
 'mirna_targets_mirdb',
 'oncogenic_signatures',
 'pid_pathways',
 'positional',
 'reactome_pathways',
 'tf_targets_gtrf',
 'tf_targets_legacy',
 'vaccine_response',
 'wikipathways'}

## GSEA

In [16]:
def get_gsea_df(results_df, comparison=""):
    # Prepare the ranking for GSEA
    ranked_genes = results_df[['log2FoldChange']].sort_values(by='log2FoldChange', ascending=False)
    
    # Prepare the ranking for GSEA
    gene_list = ranked_genes['log2FoldChange']
    gene_list.index = results_df.index  # Ensure the gene symbols are in the index
    
    # Run GSEA
    enr_gsea = dc.get_gsea_df(
        df=df,
        stat="stat",
        net=msigdb,
        source='geneset',
        target='genesymbol'
    )
    enr_gsea.to_csv(f'/home/amore/work/data/{comparison}_GSEA.csv', header=True)
    return enr_gsea

    

In [20]:
gsea_dict={}
for comparison, df in df_de_dict.items():
    enr_gsea = get_gsea_df(df,comparison=comparison)
    gsea_dict[comparison] = enr_gsea

In [ ]:
significant_GSEA = enr_gsea[enr_gsea["FDR p-value"] < 0.1].sort_values(by="NES")
significant_GSEA

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb,
    source='geneset',
    target='genesymbol',
    set_name='GOCC_ACTIN_BASED_CELL_PROJECTION'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb,
    source='geneset',
    target='genesymbol',
    set_name='MIKKELSEN_MEF_HCP_WITH_H3K27ME3'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb,
    source='geneset',
    target='genesymbol',
    set_name='HP_HEMATOLOGICAL_NEOPLASM'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb,
    source='geneset',
    target='genesymbol',
    set_name='MIR1273H_5P'
)

## ORA

In [ ]:
results_df

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kneed import KneeLocator

# Example DataFrame
df = dds

# Step 1: Sort by log2FoldChange (ascending or descending)
df = df.sort_values(by='stat', ascending=False).reset_index(drop=True)

# Step 2: Normalize values (optional)
y = np.arange(len(df))  # Indices (gene positions)
x = df['stat'].values

# Step 3: Apply Kneedle algorithm to find the elbow point
kneedle = KneeLocator(x, y, curve='convex', direction='decreasing')  # Direction depends on sorting
elbow_point = kneedle.elbow

# Plot the log2FoldChange with the detected elbow point
plt.figure(figsize=(8, 5))
plt.plot(x, y, label='stat', marker='o')
plt.axvline(elbow_point, color='red', linestyle='--', label=f'Elbow at {elbow_point}')
plt.title('Elbow Point in stat')
plt.xlabel('Index')
plt.ylabel('stat')
plt.legend()
plt.show()

In [ ]:
elbow_point

In [ ]:
### get elbow point
elbow = elbow_point

dds_de = dds[(abs(dds['stat'])>elbow) & (dds['padj']<0.1)]

dds_de.sort_values(by='log2FoldChange')

In [ ]:
top_genes = list(dds_de[dds_de['padj'] < 0.05].index)

len(top_genes)

In [ ]:
dds

In [ ]:
top_genes[:3]

In [ ]:
list(dds.index)

In [ ]:
# Infer enrichment with ora using significant deg

# Run ora
enr_pvals = dc.get_ora_df(
    df=top_genes,#[:10],
    net=msigdb,
    source='geneset',
    target='genesymbol'
)

enr_pvals.head()

In [ ]:
enr_pvals.to_csv(f'/home/amore/work/data/{experiment}_{comparison}_ORA.csv', header=True)

In [ ]:
significant_ORA = enr_pvals[(enr_pvals["p-value"] < 0.1) & (enr_pvals["FDR p-value"] < 0.15)]
significant_ORA

In [ ]:
dc.plot_dotplot(
    significant_ORA.sort_values('Combined score', ascending=False).head(15),
    x='Combined score',
    y='Term',
    s='Odds ratio',
    c='p-value',
    scale=.5,
    figsize=(5, 10)
)

### GSEA vs ORA

In [ ]:
enr_gsea[enr_gsea['Term'].isin(significant_ORA['Term'])].sort_values(by="NES")

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb[msigdb['collection']=='wikipathways'],
    source='geneset',
    target='genesymbol',
    set_name='WP_ELECTRON_TRANSPORT_CHAIN_OXPHOS_SYSTEM_IN_MITOCHONDRIA'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb[msigdb['collection']=='kegg_pathways'],
    source='geneset',
    target='genesymbol',
    set_name='KEGG_OXIDATIVE_PHOSPHORYLATION'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb[msigdb['collection']=='go_biological_process'],
    source='geneset',
    target='genesymbol',
    set_name='GOBP_PROTON_MOTIVE_FORCE_DRIVEN_ATP_SYNTHESIS'
)

In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb[msigdb['collection']=='go_biological_process'],
    source='geneset',
    target='genesymbol',
    set_name='GOBP_DNA_TEMPLATED_DNA_REPLICATION_MAINTENANCE_OF_FIDELITY'
)



In [ ]:
# Plot
dc.plot_running_score(
    df=dds_de,
    stat='stat',
    net=msigdb[msigdb['collection']=='reactome_pathways'],
    source='geneset',
    target='genesymbol',
    set_name='REACTOME_RESPIRATORY_ELECTRON_TRANSPORT_ATP_SYNTHESIS_BY_CHEMIOSMOTIC_COUPLING_AND_HEAT_PRODUCTION_BY_UNCOUPLING_PROTEINS'
)



In [ ]:
set(msigdb["collection"])